# DAAT — WACV 2027 Paper Analysis (Colab)
Reproduces **every number and figure in the submitted paper** from your Drive archives.

| Cell | What it does | Paper artifact |
|---|---|---|
| 1 | Mount Drive, unzip both archives | — |
| 2 | Aggregate all `ALL_SEQUENCES_metrics.csv` | `master_results.csv` |
| 3 | Official test-server numbers | Tables 1–2 |
| 4 | Figure 1: development progression | `figs/fig_progression.pdf` |
| 5 | Figure 2: per-sequence test results | `figs/fig_perseq_test.pdf` |
| 6 | Print LaTeX rows for Tables 2–4 | copy into Overleaf |
| 7 | *(optional)* TrackEval re-scoring of AppGallery/AppSwap/FastTracker `.txt` outputs | new gallery table |
| 8 | *(optional)* Audit-log excerpt table from a real `ids_events.csv` | Sec. 3.4 figure |

Everything is written to `Drive/.../MOT-17/PAPER1-ASSETS/`. Run top-to-bottom; cells 7–8 are independent extras.


In [ ]:
# ============================================================
# CELL 1 — Mount Drive, unzip archives
# ============================================================
# Set DATA_ROOT / OUTPUT_ROOT for your environment (Colab: mount Drive first)
import os
DATA_ROOT = os.environ.get('DATA_ROOT', '/content/data')
OUTPUT_ROOT = os.environ.get('OUTPUT_ROOT', '/content/outputs')

import os, zipfile, glob

BASE      = '${OUTPUT_ROOT}'
ZIP_TEST  = f'{BASE}/TEST-20260705T005812Z-3-001.zip'
ZIP_FINAL = f'{BASE}/FINAL-RESULTS-20260705T005803Z-3-001.zip'

# Only needed for optional CELL 7 (TrackEval):
MOT17_TRAIN_GT = '${DATA_ROOT}/MOT17/train'

WORK   = '/content/work'
ASSETS = f'{BASE}/PAPER1-ASSETS'
os.makedirs(WORK, exist_ok=True)
os.makedirs(f'{ASSETS}/figs', exist_ok=True)

for z in (ZIP_TEST, ZIP_FINAL):
    assert os.path.exists(z), f'Missing: {z}'
    print('Unzipping', os.path.basename(z), '…')
    with zipfile.ZipFile(z) as f:
        f.extractall(WORK)
print('Done. Top-level:', os.listdir(WORK))

In [ ]:
# ============================================================
# CELL 2 — Aggregate every ALL_SEQUENCES_metrics.csv -> master table
# ============================================================
import pandas as pd
pd.set_option('display.width', 200)

rows = []
for p in sorted(glob.glob(f'{WORK}/**/ALL_SEQUENCES_metrics.csv', recursive=True)):
    try:
        df = pd.read_csv(p)
        r = df[df['sequence'].astype(str).str.contains('AVERAGE-ALL|COMBINED', na=False)]
        r = (df.tail(1) if r.empty else r).iloc[-1]
        rows.append(dict(
            exp=p.replace(WORK + '/', '').replace('/ALL_SEQUENCES_metrics.csv',''),
            HOTA=r.get('HOTA'), MOTA=r.get('MOTA'), IDF1=r.get('IDF1'),
            DetA=r.get('DetA'), AssA=r.get('AssA'), IDs=r.get('IDs'), Frag=r.get('Frag')))
    except Exception as e:
        print('skip', p, e)

master = pd.DataFrame(rows).sort_values('exp').reset_index(drop=True)
master.to_csv(f'{ASSETS}/master_results.csv', index=False)
print(master.to_string())
print('\nSaved ->', f'{ASSETS}/master_results.csv')

def get(key):
    m = master[master.exp.str.contains(key, regex=False)]
    return m.iloc[0] if len(m) else None

In [ ]:
# ============================================================
# CELL 3 — Official MOT17 test-server results (Tables 1–2)
# Verify these decimals against your MOTChallenge account page.
# ============================================================
test_server = pd.DataFrame([
    # seq,        HOTA,   MOTA,   IDF1,   IDs,  Frag
    ('MOT17-01', 48.46, 52.14, 59.25,  54, 116),
    ('MOT17-03', 68.52, 91.09, 85.74, 172, 527),
    ('MOT17-06', 51.95, 64.69, 64.77,  63, 243),
    ('MOT17-07', 50.51, 71.38, 62.73,  98, 327),
    ('MOT17-08', 41.30, 59.88, 46.57, 286, 539),
    ('MOT17-12', 57.70, 55.51, 69.03,  32, 180),
    ('MOT17-14', 48.98, 57.26, 66.93,  80, 361),
    ('Combined', 60.64, 77.86, 75.19, 2355, 6879),
], columns=['seq','HOTA','MOTA','IDF1','IDs','Frag'])
test_server.to_csv(f'{ASSETS}/test_server_results.csv', index=False)
test_server

In [ ]:
# ============================================================
# CELL 4 — Figure 1: development progression on ALL-21
# (identical styling to the figure in the compiled paper)
# ============================================================
import matplotlib
matplotlib.rcParams.update({'font.size': 8, 'pdf.fonttype': 42, 'font.family': 'serif'})
import matplotlib.pyplot as plt

ALL21 = [
    ('v1 baseline',            'Test-6-YOLOX/ALL-21'),
    ('v2 motion\ntie-break',   'v2-motion-tiebreak'),
    ('v3 occl.\ngating',       'v3-occlusion'),
    ('stage-1\nfusion',        'Test-8-YOLOX/ALL-21-stage1-fusion'),
    ('v5 long-term\nre-assoc.','v5-longterm-reid'),
    ('v6 w/h\nKalman',         'v6-whkalman'),
    ('v8\nvisibility',         'v8-visible'),
    ('v8-ws\nwtd. sum',        'wsum-only'),
    ('v9 Fast-\nTracker',      'v9-fasttracker'),
    ('v11 relaxed\ngate',      'relax-gate'),
    ('CLIP-ReID',              'CLIPREID'),
    ('diff-\nmargin',          'diffmargin'),
]

labels, hotas, ids_ = [], [], []
for lab, key in ALL21:
    r = get(key)
    if r is not None:
        labels.append(lab); hotas.append(r.HOTA); ids_.append(r.IDs)

fig, ax1 = plt.subplots(figsize=(6.9, 2.4))
x = range(len(labels))
ax1.bar(x, hotas, color='#35618f', width=0.62)
ax1.set_ylabel('HOTA'); ax1.set_ylim(63, 66.6)
ax1.axhline(hotas[0], color='#35618f', ls=':', lw=0.8)
ax2 = ax1.twinx()
ax2.plot(x, ids_, 'o-', color='#b02418', lw=1.1, ms=3)
ax2.set_ylabel('mean ID switches', color='#b02418'); ax2.tick_params(axis='y', colors='#b02418')
ax1.set_xticks(list(x)); ax1.set_xticklabels(labels, fontsize=6.5)
for a in (ax1, ax2): a.spines['top'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{ASSETS}/figs/fig_progression.pdf')
plt.show()

In [ ]:
# ============================================================
# CELL 5 — Figure 2: per-sequence MOT17 test results
# ============================================================
ts = test_server[test_server.seq != 'Combined']
fig, ax = plt.subplots(figsize=(3.3, 2.2))
w = 0.27; x = range(len(ts))
ax.bar([i-w for i in x], ts.HOTA, width=w, label='HOTA', color='#35618f')
ax.bar(list(x),          ts.MOTA, width=w, label='MOTA', color='#8ea9c4')
ax.bar([i+w for i in x], ts.IDF1, width=w, label='IDF1', color='#c9a13b')
ax.set_xticks(list(x)); ax.set_xticklabels(ts.seq, rotation=40, ha='right', fontsize=6.5)
ax.set_ylim(0, 100); ax.legend(frameon=False, fontsize=7, ncol=3, loc='upper center')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{ASSETS}/figs/fig_perseq_test.pdf')
plt.show()

In [ ]:
# ============================================================
# CELL 6 — LaTeX rows for the paper's tables (copy into Overleaf)
# ============================================================
def row(label, r, cols=('HOTA','MOTA','IDF1','AssA','IDs')):
    vals = ' & '.join(f'{r[c]:.2f}' if c != 'IDs' else f'{r[c]:.1f}' for c in cols)
    return f'{label} & {vals} \\\\'

print('% ---- Table 3: component study on ALL-21 ----')
for label, key in ALL21:
    r = get(key)
    if r is not None:
        print(row(label.replace('\n',' '), r))

VALHALF = [
    ('baseline (val-half)',  'VAL-HALF/'),
    ('+ DLOW logging',       'VAL-HALF-DLOW'),
    ('+ OCC+DLOW (ungated)', 'VAL-HALF-OCCDLOW/'),
    ('v9',                   'VAL-HALF-v9-fasttracker'),
    ('v9 park-only',         'v9-parkonly'),
    ('v9 s2-iou65',          'v9-s2iou65'),
    ('v9 stage-2-only app.', 'v9-stage2only'),
]
print('\n% ---- Table 4: val-half ablations ----')
for label, key in VALHALF:
    r = get(key)
    if r is not None:
        print(row(label, r, cols=('HOTA','MOTA','IDF1','IDs')))

print('\n% ---- Table 2: per-sequence test ----')
for _, r in test_server.iterrows():
    print(f"{r['seq']} & {r['HOTA']:.1f} & {r['MOTA']:.1f} & {r['IDF1']:.1f} & {int(r['IDs'])} & {int(r['Frag'])} \\\\")

## CELL 7 *(optional)* — Re-score AppGallery / AppSwap / FastTracker raw outputs
Those folders contain MOT-format `.txt` results with no metric CSVs. This scores them with **TrackEval** against MOT17 train GT — the numbers become the new *appearance-gallery & buffer-size* table (you have ~3 pages of headroom for it). Requires `MOT17_TRAIN_GT` to be valid.


In [ ]:
# ============================================================
# CELL 7 (corrected) — TrackEval re-evaluation, two-pass
# Pass A: trackers with all 21 seq-det files -> full protocol
# Pass B: buffer sweeps on their COMMON subset (7 FRCNN pairs),
#         with full-coverage trackers re-scored on that subset
#         for direct comparability.
# Fixes vs. previous version:
#   - numpy compatibility shim (TrackEval uses removed np.float/np.int/np.bool)
#   - USE_PARALLEL=False (the crash occurred inside pool workers; serial is
#     robust and 21 sequences take only a few minutes)
# ============================================================
!pip install -q git+https://github.com/JonathonLuiten/TrackEval.git

import shutil, os
import numpy as np
import pandas as pd

# --- compatibility shim: MUST come before trackeval is used ---
np.float = float
np.int   = int
np.bool  = bool

import trackeval

APP_FOLDERS = {
    'FastTracker-REPRO': f'{WORK}/TEST/FastTracker-REPRO/train',
    'AppGallery':        f'{WORK}/TEST/AppGalleryTracker/train',
    'AppSwap':           f'{WORK}/TEST/AppSwapTracker/train',
    'AppSwap_b010':      f'{WORK}/TEST/AppSwap_b010/train',
    'AppSwap_b015':      f'{WORK}/TEST/AppSwap_b015/train',
    'AppSwap_b020':      f'{WORK}/TEST/AppSwap_b020/train',
    'AppSwap_b025':      f'{WORK}/TEST/AppSwap_b025/train',
    'AppGallery_b015':   f'{WORK}/TEST/AppGallery_b015/train',
    'AppGallery_b020':   f'{WORK}/TEST/AppGallery_b020/train',
    'AppGallery_b025':   f'{WORK}/TEST/AppGallery_b025/train',
    'AppGallery_b030':   f'{WORK}/TEST/AppGallery_b030/train',
}

# ---- discover per-tracker coverage ----
coverage = {}
for name, fold in APP_FOLDERS.items():
    if os.path.isdir(fold):
        coverage[name] = {os.path.splitext(f)[0] for f in os.listdir(fold) if f.endswith('.txt')}
full   = {n for n, s in coverage.items() if len(s) == 21}
sweeps = {n for n in coverage if n not in full}
common = set.intersection(*[coverage[n] for n in sweeps]) if sweeps else set()
print('Full-coverage trackers :', sorted(full))
print('Sweep trackers         :', sorted(sweeps))
print(f'Common sweep subset ({len(common)}):', sorted(common))

def run_eval(pass_name, trackers, seqs):
    """Fresh TrackEval layout for `trackers` restricted to `seqs`; returns combined rows."""
    EV = f'/content/EVAL_{pass_name}'
    shutil.rmtree(EV, ignore_errors=True)
    GT = f'{EV}/gt/mot_challenge/MOT17-train'
    TR = f'{EV}/trackers/mot_challenge/MOT17-train'
    SM = f'{EV}/gt/mot_challenge/seqmaps'
    for d in (GT, TR, SM):
        os.makedirs(d, exist_ok=True)
    seqs = sorted(seqs)
    for s in seqs:
        os.makedirs(f'{GT}/{s}/gt', exist_ok=True)
        shutil.copy(f'{MOT17_TRAIN_GT}/{s}/gt/gt.txt', f'{GT}/{s}/gt/gt.txt')
        shutil.copy(f'{MOT17_TRAIN_GT}/{s}/seqinfo.ini', f'{GT}/{s}/seqinfo.ini')
    with open(f'{SM}/MOT17-train.txt', 'w') as f:
        f.write('name\n' + '\n'.join(seqs) + '\n')
    for name in trackers:
        os.makedirs(f'{TR}/{name}/data', exist_ok=True)
        for s in seqs:
            shutil.copy(f'{APP_FOLDERS[name]}/{s}.txt', f'{TR}/{name}/data/{s}.txt')

    ev_cfg = trackeval.Evaluator.get_default_eval_config()
    ev_cfg.update(dict(PRINT_CONFIG=False, TIME_PROGRESS=False, USE_PARALLEL=False))
    ds_cfg = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    ds_cfg.update(dict(GT_FOLDER=f'{EV}/gt/mot_challenge',
                       TRACKERS_FOLDER=f'{EV}/trackers/mot_challenge',
                       BENCHMARK='MOT17', SPLIT_TO_EVAL='train',
                       DO_PREPROC=True, PRINT_CONFIG=False,
                       SEQMAP_FILE=f'{SM}/MOT17-train.txt'))
    res, _ = trackeval.Evaluator(ev_cfg).evaluate(
        [trackeval.datasets.MotChallenge2DBox(ds_cfg)],
        [trackeval.metrics.HOTA(), trackeval.metrics.CLEAR(), trackeval.metrics.Identity()])
    rows = []
    for name in trackers:
        r = res['MotChallenge2DBox'][name]['COMBINED_SEQ']['pedestrian']
        rows.append(dict(protocol=pass_name, exp=name, n_seqs=len(seqs),
            HOTA=100*r['HOTA']['HOTA'].mean(), AssA=100*r['HOTA']['AssA'].mean(),
            DetA=100*r['HOTA']['DetA'].mean(), MOTA=100*r['CLEAR']['MOTA'],
            IDF1=100*r['Identity']['IDF1'],   IDs=int(r['CLEAR']['IDSW'])))
    return rows

# ---- Pass A: full-coverage trackers on all 21 pairs ----
rows = []
if full:
    rows += run_eval('FULL21', sorted(full), coverage[next(iter(full))])

# ---- Pass B: everyone on the common sweep subset ----
if common:
    rows += run_eval('SUBSET', sorted(full | sweeps), common)
else:
    print('WARNING: sweep trackers share no common sequences — Pass B skipped.')

app_table = pd.DataFrame(rows)
app_table.to_csv(f'{ASSETS}/appearance_module_results.csv', index=False)
print(app_table.to_string())

print('\n% ---- LaTeX rows: full 21-pair protocol ----')
for _, r in app_table[app_table.protocol == 'FULL21'].iterrows():
    print(f"{r['exp'].replace('_',' ')} & {r.HOTA:.2f} & {r.MOTA:.2f} & {r.IDF1:.2f} & {r.AssA:.2f} & {r.IDs} \\\\")

print(f"\n% ---- LaTeX rows: buffer sweep, common subset ({len(common)} seq-det pairs, FRCNN) ----")
for _, r in app_table[app_table.protocol == 'SUBSET'].iterrows():
    print(f"{r['exp'].replace('_',' ')} & {r.HOTA:.2f} & {r.MOTA:.2f} & {r.IDF1:.2f} & {r.AssA:.2f} & {r.IDs} \\\\")

## CELL 8 *(optional)* — Audit-log excerpt for the paper (Sec. 3.4)
Pulls a few rows from a real `ids_events.csv` produced by your tracker and prints them as a compact LaTeX table — concrete evidence for reviewers that the audit layer exists and is usable.


In [ ]:
# ============================================================
# CELL 8 — Audit-log excerpt table
# ============================================================
cands = sorted(glob.glob(f'{WORK}/**/ids_events.csv', recursive=True)) or \
        sorted(glob.glob(f'{WORK}/**/*_ids_events.csv', recursive=True))
print(f'{len(cands)} audit logs found'); print('\n'.join(cands[:5]))

if cands:
    log = pd.read_csv(cands[0])
    print('\nColumns:', list(log.columns)[:14])
    ex = log.head(4)
    show = [c for c in ('frame_id','track_id','event','iou','app_sim','score') if c in log.columns]
    show = show or list(log.columns)[:6]
    print('\n% ---- LaTeX rows: audit-log excerpt ----')
    for _, r in ex.iterrows():
        cells = []
        for c in show:
            v = r[c]
            cells.append(f'{v:.3f}' if isinstance(v, float) else str(v)[:18])
        print(' & '.join(cells) + ' \\\\')
    ex.to_csv(f'{ASSETS}/audit_log_excerpt.csv', index=False)

In [ ]:
# ============================================================
# CELL 9 (corrected) — Flag-calibration: do flagged-ambiguous
# events predict real (GT) identity errors?
# Uses train-run audit logs (Test-14-CLIPREID, all 21 pairs) +
# MOT17 train GT via motmetrics.
#
# Output to read: the final ratio line. Flagged events landing
# near GT switches at ~2x+ the unflagged rate = calibrated flag.
# ============================================================
!pip install -q motmetrics

import numpy as np
# --- NumPy 2.0 compatibility shims (MUST precede motmetrics import) ---
np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)
np.float = float; np.int = int; np.bool = bool

import motmetrics as mm
import pandas as pd, glob, os

LOG_DIR  = f'{WORK}/TEST/Test-14-YOLOX-CLIPREID/ALL-21'
RES_DIRS = [f'{WORK}/TEST/Test-14-YOLOX-CLIPREID/ALL-21',
            f'{WORK}/TEST/Test-14-YOLOX-CLIPREID']
WINDOW = 5   # frames around an event within which a GT switch counts as "at" the event

# ---- 0. Inspect the log vocabulary FIRST (schemas differ across runs) ----
log_paths = sorted(glob.glob(f'{LOG_DIR}/*_ids_events.csv'))
print(f'{len(log_paths)} audit logs found in {LOG_DIR}')
assert log_paths, 'No ids_events logs found — check LOG_DIR.'

probe = pd.read_csv(log_paths[0])
print('Columns:', list(probe.columns))
print('\nreason vocabulary (first log):')
print(probe.reason.astype(str).str.split(' ').str[0].value_counts())

# ---- flag definition ----
# Flagged-ambiguous = refusal events (margin/gate failures). If this run's
# vocabulary differs, the fallback flags low-margin accepted events instead.
FLAG_PREFIXES = ('margin_failed', 'gate_failed')
has_refusals = probe.reason.astype(str).str.startswith(FLAG_PREFIXES).any()
if not has_refusals and 'cosine_margin' in probe.columns:
    print('\nNOTE: no refusal reasons in this run; falling back to '
          'low-margin flagging (cosine_margin < 0.05).')

def split_flagged(log):
    """Return (flagged_events, unflagged_appearance_events) for one log."""
    reasons = log.reason.astype(str)
    if reasons.str.startswith(FLAG_PREFIXES).any():
        is_flag = reasons.str.startswith(FLAG_PREFIXES)
    elif 'cosine_margin' in log.columns and log.cosine_margin.notna().any():
        is_flag = log.cosine_margin < 0.05
    else:
        return log.iloc[0:0], log.iloc[0:0]   # nothing usable in this log
    has_app = log.cosine_sim.notna() if 'cosine_sim' in log.columns else ~is_flag
    return log[is_flag], log[~is_flag & has_app]

def find_result_txt(seq):
    for d in RES_DIRS:
        hits = glob.glob(f'{d}/**/{seq}.txt', recursive=True)
        if hits:
            return hits[0]
    return None

# ---- 1. Per-sequence: GT switch events via motmetrics, then co-occurrence ----
rows, skipped = [], []
for log_path in log_paths:
    seq = os.path.basename(log_path).replace('_ids_events.csv', '')
    res_path = find_result_txt(seq)
    gt_path  = f'{MOT17_TRAIN_GT}/{seq}/gt/gt.txt'
    if res_path is None or not os.path.exists(gt_path):
        skipped.append(seq)
        continue

    gt  = mm.io.loadtxt(gt_path,  fmt='mot15-2D', min_confidence=1)
    ts  = mm.io.loadtxt(res_path, fmt='mot15-2D')
    acc = mm.utils.compare_to_groundtruth(gt, ts, 'iou', distth=0.5)
    ev  = acc.events.reset_index()
    sw  = ev[ev.Type == 'SWITCH'][['FrameId', 'HId']].values  # (frame, tracker_id)

    log = pd.read_csv(log_path)
    flagged, unflagged = split_flagged(log)
    for is_flag, sub in [(True, flagged), (False, unflagged)]:
        hit = 0
        for _, e in sub.iterrows():
            ids = {e.old_id, e.new_id}
            near = any(abs(f - e.frame) <= WINDOW and h in ids for f, h in sw)
            hit += near
        rows.append(dict(seq=seq, flagged=is_flag,
                         n_events=len(sub), n_near_switch=hit))

if skipped:
    print(f'\nSkipped {len(skipped)} sequences (missing result txt or GT): {skipped[:5]}...')

# ---- 2. Aggregate ----
cal = pd.DataFrame(rows)
cal.to_csv(f'{ASSETS}/flag_calibration.csv', index=False)
summary = cal.groupby('flagged')[['n_events', 'n_near_switch']].sum()
summary['rate'] = summary.n_near_switch / summary.n_events.clip(lower=1)
print('\n', summary)

if True in summary.index and False in summary.index and summary.loc[True, 'n_events'] > 0:
    r_f, r_u = summary.loc[True, 'rate'], summary.loc[False, 'rate']
    print(f"\nRESULT: flagged events co-occur with a GT identity switch at "
          f"{r_f:.1%} vs {r_u:.1%} for unflagged accepted events "
          f"-> ratio {r_f / max(r_u, 1e-9):.1f}x "
          f"(flagged n={int(summary.loc[True,'n_events'])}, "
          f"unflagged n={int(summary.loc[False,'n_events'])})")
else:
    print('\nRESULT: no flagged events found under either definition — '
          'paste the reason vocabulary above and we will adapt the flag definition.')

## Done
`PAPER1-ASSETS/` in your Drive now holds: `master_results.csv`, `test_server_results.csv`, `figs/fig_progression.pdf`, `figs/fig_perseq_test.pdf`, and (if cells 7–8 ran) `appearance_module_results.csv` + `audit_log_excerpt.csv`.

In Overleaf, upload the two PDFs into the project's `figs/` folder (overwriting the ones shipped in the zip) — the paper references them by those exact names.
